# Assignment 9: Cloud type classification from satellite imagery with CNNs

Cloud classification is a natural convolutional problem. Cloud types are defined by *texture
and spatial organization* — whether the cloud is wispy or lumpy, layered or towering,
scattered or continuous — rather than by the brightness of any individual pixel. A fully
connected network throws that spatial structure away; a convolutional network is built to
exploit it.

Getting cloud type right matters for climate: different cloud types have opposite effects on
the radiation budget. High thin cirrus traps outgoing longwave radiation and warms; low thick
stratocumulus reflects incoming sunlight and cools. Cloud feedbacks remain one of the largest
sources of uncertainty in climate sensitivity estimates.

You will use the **CCSN (Cirrus Cumulus Stratus Nimbus) database**: 2,543 real sky images
labelled into 11 categories following the World Meteorological Organization genus
classification.

Answer each numbered question in the empty cell below it.

```{admonition} A note on the imagery
:class: note
CCSN is **ground-based** sky imagery rather than top-of-atmosphere satellite imagery. The
convolutional methods are identical, and the dataset is small enough to train on in an
afternoon. If you would like to work with true satellite imagery instead, the Kaggle
competition `understanding_cloud_organization` provides labelled MODIS scenes — it is a
several-gigabyte download, so plan accordingly.
```

In [ ]:
!pip install kagglehub

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     RandomFlip, RandomRotation, RandomZoom, Rescaling)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
import kagglehub

path = kagglehub.dataset_download("mmichelli/cirrus-cumulus-stratus-nimbus-ccsn-database")
print("Path to dataset files:", path)
os.listdir(path)

The 11 CCSN classes use standard abbreviations:

| Code | Cloud type | | Code | Cloud type |
| --- | --- | --- | --- | --- |
| Ci | Cirrus | | Ac | Altocumulus |
| Cs | Cirrostratus | | As | Altostratus |
| Cc | Cirrocumulus | | Ns | Nimbostratus |
| Cu | Cumulus | | Sc | Stratocumulus |
| Cb | Cumulonimbus | | St | Stratus |
| Ct | Contrail | | | |

## Part 1: Load and inspect

1) Build training and validation datasets with
`keras.utils.image_dataset_from_directory`, using an 80/20 split, `image_size=(128, 128)`
and `seed=0`. Report the class names it finds.

2) Display one example image from each of the 11 classes in a grid, labelled.

3) Count the images in each class. Is the dataset balanced? Which classes are rarest, and what will that do to your accuracy metric?

## Part 2: A first CNN

4) Build a convolutional network with three `Conv2D` + `MaxPooling2D` blocks followed by a
dense classifier head. Start with a `Rescaling(1./255)` layer. Print the model summary and
report the number of trainable parameters.

5) Compile and train for around 20 epochs, keeping the training history. Plot training and validation accuracy against epoch.

6) Does the gap between training and validation accuracy widen as training proceeds? What is that gap called, and what does it tell you?

*Write your answer here.*

## Part 3: Data augmentation

Sky images have no preferred orientation — a cumulus cloud photographed upside down is still
a cumulus cloud. That makes augmentation both safe and useful here.

7) Add an augmentation block (`RandomFlip`, `RandomRotation`, `RandomZoom`) before the
convolutional layers, and retrain. Plot the new accuracy curves against the originals.

8) Did augmentation help? Report the change in validation accuracy, and explain in terms of what augmentation does to the effective size of the training set.

9) Name one augmentation that would be **inappropriate** for this dataset, and explain what it would teach the model that is not true.

*Write your answer here.*

## Part 4: Transfer learning

With 2,543 images you cannot train a large network from scratch. Transfer learning reuses
features learned on a much larger dataset.

10) Load a pretrained backbone (`keras.applications.MobileNetV2` with
`weights="imagenet", include_top=False`), freeze it, and attach your own classifier head.
Train and report validation accuracy.

11) Compare against your from-scratch model. How much did transfer learning gain you, and how did training time compare?

12) ImageNet contains almost no sky imagery. Explain why features learned on photographs of everyday objects transfer usefully to clouds anyway.

*Write your answer here.*

## Part 5: Where it fails

13) Produce a confusion matrix on the validation set for your best model, with class names
on both axes.

14) Which pairs of cloud types are confused most often? Look up what those types actually look like and explain whether the confusion is meteorologically reasonable.

*Write your answer here.*

15) Display a grid of misclassified images with their true and predicted labels. Would you have classified them correctly yourself?

16) Report per-class precision and recall. Relate the weakest classes back to your class counts from question 3.

## Part 6: Interpretation

17) Visualize the filters in your first convolutional layer, or the feature maps they produce
for one input image. What kinds of structure is the first layer responding to?

18) These images were collected from a limited number of locations and cameras. If you
deployed this model on imagery from a different sky camera in a different climate zone,
what would you expect to happen, and how would you test it before trusting the output?

*Write your answer here.*

19) Suppose this classifier were used to build a long-term record of cloud type frequency for climate monitoring. What property would the model need that validation accuracy does not measure?

*Write your answer here.*